# Inference Workflow with DLOmix

How to run a trained model on **new, unlabelled peptides** without rebuilding a dataset or
remembering the training preprocessing. Two pieces do the work:

- **`PeptidePreprocessor`** — reproduces the exact training-time preprocessing (alphabet,
  encoding, padding, features) and turns raw sequences into model-ready tensors.
- **`InferencePipeline`** — bundles a model with its preprocessor for one-call `predict` and
  a single save/load artifact.

We cover three options: (1) inference right after training, (2) saving a bundle, and
(3) inference-only in a fresh session with no training and no dataset.

> The backend (TensorFlow/PyTorch) is chosen from `DLOMIX_BACKEND` **before** importing
> dlomix. This notebook uses the default (TensorFlow).

In [1]:
# import os; os.environ['DLOMIX_BACKEND'] = 'tensorflow'  # set before importing dlomix
import numpy as np
from datasets import Dataset

from dlomix.data import RetentionTimeDataset, PeptidePreprocessor
from dlomix.models import PrositRetentionTimePredictor
from dlomix.pipelines import InferencePipeline

Using TensorFlow Backend for DLOmix. To change the backend, set the DLOMIX_BACKEND environment variable to tensorflow or pytorch and re-import DLOmix.


## 1. A trained model + dataset (mocked)

Replace this cell with your real data and training. Here we build a tiny in-memory dataset
and run a 1-epoch fit just so the model exists.

In [2]:
# mock data: peptide sequences + a retention-time label
seqs = ['ACDEFGHIK', 'PEPTIDEK', 'MKLVAAR', 'GGGGSSSK', 'ACDEFGHIKLMN', 'PEPK'] * 8
data = {'modified_sequence': seqs, 'indexed_retention_time': [float(len(s)) for s in seqs]}

dataset = RetentionTimeDataset(
    data_source=Dataset.from_dict(data),
    data_format='hf',
    sequence_column='modified_sequence',
    label_column='indexed_retention_time',
    val_ratio=0.2,
    max_seq_len=20,
    batch_size=8,
)

model = PrositRetentionTimePredictor(seq_length=22, alphabet=dataset.extended_alphabet)
model.compile(optimizer='adam', loss='mse')
model.fit(dataset.tensor_train_data, validation_data=dataset.tensor_val_data,
          epochs=1, verbose=0)  # mock training

/Users/Omar/Documents/VSCode_repos/dlomix/dlomix/src/dlomix/data/loading.py:117: UserWarning: data_format="hf" with a Dataset: the dataset will be automatically split into train/val (and optionally test) according to the split configuration. val_data_source and test_data_source are ignored.
  warnings.warn(
/Users/Omar/Documents/VSCode_repos/dlomix/dlomix/src/dlomix/data/processing/chain.py:64: UserWarning: Encoding scheme is EncodingScheme.UNMOD, this enforces removing all occurences of PTMs in the sequences. If you prefer to encode the (amino-acids)+PTM combinations as tokens in the vocabulary, please use the encoding scheme 'naive-mods'.
  warnings.warn(
/Users/Omar/Documents/VSCode_repos/dlomix/dlomix/src/dlomix/data/processing/processors.py:390: UserWarning: The unknown token 'X' is not present in the provided alphabet. It will be added with index 1.
  warnings.warn(


Mapping SequenceParsingProcessor on split train (num_proc=10):   0%|          | 0/38 [00:00<?, ? examples/s]

Mapping SequenceParsingProcessor on split val (num_proc=10):   0%|          | 0/10 [00:00<?, ? examples/s]

Mapping SequencePTMRemovalProcessor on split train (num_proc=10):   0%|          | 0/38 [00:00<?, ? examples/s…

Mapping SequencePTMRemovalProcessor on split val (num_proc=10):   0%|          | 0/10 [00:00<?, ? examples/s]

...

Mapping SequencePaddingProcessor on split train (num_proc=10):   0%|          | 0/38 [00:00<?, ? examples/s]

Filter (num_proc=10):   0%|          | 0/38 [00:00<?, ? examples/s]

Mapping SequencePaddingProcessor on split val (num_proc=10):   0%|          | 0/10 [00:00<?, ? examples/s]

Filter (num_proc=10):   0%|          | 0/10 [00:00<?, ? examples/s]

2026-06-23 15:16:34.353025: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Max
2026-06-23 15:16:34.353064: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2026-06-23 15:16:34.353071: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 12.48 GB
2026-06-23 15:16:34.353108: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-06-23 15:16:34.353125: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
/Users/Omar/miniconda3/envs/dlx/lib/python3.11/site-packages/datasets/arrow_dataset.py:405: FutureWarning: The output of `to_tf_dataset` will change when a passing single element li

## 2. Inference right after training (in-notebook)

**Option A — preprocessor + `model.predict`.** Get the preprocessor from the dataset and
call it on raw sequences; the output is exactly what the model consumes.

In [ ]:
#model.predict(prep(new_data))

In [3]:
prep = dataset.get_preprocessor()

tensors = prep(['ACDEM[UNIMOD:35]K', 'PEPTIDEK'])   # raw sequences -> tensors
predictions = model.predict(tensors, verbose=0)
predictions.ravel()

/Users/Omar/Documents/VSCode_repos/dlomix/dlomix/src/dlomix/data/processing/processors.py:368: UserWarning: The unknown token 'X' is already present in the provided alphabet with index 1. If you prefer the default behavior, consider removing it from the alphabet and it will have the default index of 1.
  warnings.warn(


Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

array([4.8511567, 4.3735776], dtype=float32)

**Option B — `InferencePipeline`.** Bundle the model with its preprocessor and predict on
raw inputs in one call.

In [4]:
pipeline = InferencePipeline.from_model_and_dataset(model, dataset)
pipeline.predict(['ACDEM[UNIMOD:35]K', 'PEPTIDEK']).ravel()

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

1/1 [==============================] - 0s 25ms/step


array([4.8511567, 4.3735776], dtype=float32)

## 3. Save the bundle for later / sharing

One directory holds the preprocessor, model, and metadata.

In [5]:
pipeline.save('artifacts/rt_model', overwrite=True)

'artifacts/rt_model'

## 4. Inference-only — load and predict, no training, no dataset

Imagine a **fresh Python session**: only the saved artifact is needed. `load` restores the
model + preprocessor and checks they match.

In [3]:
pipeline2 = InferencePipeline.load('artifacts/rt_model')
pipeline2.predict(['ACDEFGHIK', 'MKLVAAR', 'PEPK']).ravel()

/Users/Omar/Documents/VSCode_repos/dlomix/dlomix/src/dlomix/data/processing/processors.py:377: UserWarning: The unknown token 'X' is already present in the provided alphabet with index 1. If you prefer the default behavior, consider removing it from the alphabet and it will have the default index of 1.
  warnings.warn(


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

1/1 [==============================] - 1s 560ms/step


array([4.2983775, 4.8189893, 5.3660126], dtype=float32)

## 5. Input formats & the standalone preprocessor

`predict`/`transform` accept a single string, a list/numpy array, a dict that also carries
`model_features` (e.g. `collision_energy`), a pandas DataFrame, or a HuggingFace `Dataset`.

In [7]:
import pandas as pd

prep('ACDEFGHIK')                                   # single string
prep(np.array(['ACDEFGHIK', 'PEPK']))               # numpy array
prep(pd.DataFrame({'modified_sequence': ['PEPK']})) # DataFrame
# prep({'modified_sequence': [...], 'collision_energy': [...]})  # dict with model features
print('ok')

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

ok


The preprocessor can also live on its own — rebuilt from a saved dataset directory, or
saved/loaded as a small standalone artifact to ship next to a model.

In [8]:
# from a dataset saved via dataset.save_to_disk(...)
# prep = PeptidePreprocessor.from_saved('processed_datasets/rt_dataset')

prep.save('artifacts/rt_preprocessor')
prep = PeptidePreprocessor.load('artifacts/rt_preprocessor')
prep(['PEPTIDEK'])

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

<_PrefetchDataset element_spec=TensorSpec(shape=(None, 22), dtype=tf.int64, name=None)>

## 6. Share on the HuggingFace Hub

The bundle is a self-contained folder, so pushing it to the Hub is one call — model,
preprocessor, metadata, and an auto-generated model card travel together. Loading anywhere
is then `from_pretrained`, no training or dataset needed.

In [ ]:
# requires HF auth once: `huggingface-cli login`

# push the bundle to a Hub repo
model_hub_id = "omsh/test-model"
pipeline2.push_to_hub(model_hub_id, private=True)

'omsh/test-model'

In [ ]:
# load it back anywhere — no training, no dataset, just the repo id
reloaded = InferencePipeline.from_pretrained(model_hub_id)
reloaded.predict(['ACDEFGHIK', 'MKLVAAR', 'PEPK']).ravel()

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

1/1 [==============================] - 1s 530ms/step


array([4.2983775, 4.8189893, 5.3660126], dtype=float32)

**Notes**

- On construct/load, the pipeline checks the model's embedding size matches the alphabet and
  raises on mismatch — guarding against pairing the wrong model and preprocessor.
- Custom *callable* feature extractors can't be serialized: kept by `get_preprocessor()`,
  dropped by `save()`/`from_saved()` with a warning. Built-in feature names always restore.
- A pipeline must be loaded under the same `DLOMIX_BACKEND` it was saved with.